# 🥉 Bronze Layer — GH Archive Batch Ingestion

This notebook ingests seven days of raw GitHub event data from the
AWS S3 landing zone into the Bronze layer of the lakehouse.

## Data Flow

```text
GH Archive
     ↓
Python Batch Ingestion
     ↓
AWS S3 Raw Layer
     ↓
Apache Spark
     ↓
Bronze Delta Table

##Imports and configuration

In [0]:
# ============================================================
# IMPORTS
# ============================================================

from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    count,
    countDistinct
)


# ============================================================
# CONFIGURATION
# ============================================================

RAW_BASE_PATH = (
    "s3://tahmid-gharchive-lakehouse/raw/github/"
)

BRONZE_TABLE = (
    "github_lakehouse.bronze.github_events_raw"
)

YEAR = "2024"
MONTH = "02"

# Process February 1 through February 7
DAYS_TO_PROCESS = range(1, 8)


print("=" * 65)
print("⚙️  BRONZE INGESTION CONFIGURATION")
print("=" * 65)

print(f"☁️  Source : {RAW_BASE_PATH}")
print(f"🥉 Target : {BRONZE_TABLE}")
print("📅 Range  : 2024-02-01 → 2024-02-07")
print("📦 Days   : 7")

print("=" * 65)

⚙️  BRONZE INGESTION CONFIGURATION
☁️  Source : s3://tahmid-gharchive-lakehouse/raw/github/
🥉 Target : github_lakehouse.bronze.github_events_raw
📅 Range  : 2024-02-01 → 2024-02-07
📦 Days   : 7


# 🗃️ Create the Bronze Table

The Bronze table schema is created explicitly before ingestion begins.

Defining the schema first prevents different batches from accidentally
creating different table structures.

The table is stored as a Unity Catalog managed Delta table.

##Create catalog, schema and Bronze table

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS github_lakehouse;

CREATE SCHEMA IF NOT EXISTS github_lakehouse.bronze;


CREATE TABLE IF NOT EXISTS github_lakehouse.bronze.github_events_raw
(
    raw_json STRING,

    _source_file STRING,

    _source_file_name STRING,

    _source_date DATE,

    _ingested_at TIMESTAMP
)
USING DELTA;

##Check the table schema

In [0]:
print("🥉 Bronze table ready.\n")

spark.sql(
    f"DESCRIBE TABLE {BRONZE_TABLE}"
).show(
    truncate=False
)

🥉 Bronze table ready.

+-----------------+---------+-------+
|col_name         |data_type|comment|
+-----------------+---------+-------+
|raw_json         |string   |NULL   |
|_source_file     |string   |NULL   |
|_source_file_name|string   |NULL   |
|_source_date     |date     |NULL   |
|_ingested_at     |timestamp|NULL   |
+-----------------+---------+-------+



# 🔄 Daily Batch Ingestion

The dataset is processed one day at a time.

For every day:

1. Build the corresponding S3 path.
2. Read all 24 hourly gzip archives.
3. Preserve each GitHub event as raw JSON.
4. Add ingestion metadata.
5. Check whether the day has already been loaded.
6. Append the batch to the Bronze Delta table.
7. Validate the number of events and source files.

### Rerun Protection

Before writing a daily batch, the notebook checks whether that date
already exists in Bronze.

If it does, the batch is skipped.

This prevents accidental duplicate ingestion when the notebook is
rerun.

### Error Handling

Spark transformations are lazy.

Therefore, DataFrame transformations are defined before the
`try/except` block.

The actual Spark actions — such as Delta writes and validation queries —
are executed inside `try/except`, where their failures can be caught.

In [0]:
# ============================================================
# INGESTION STATISTICS
# ============================================================

successful_days = 0
skipped_days = 0
failed_days = 0

failed_batches = []


total_days = len(DAYS_TO_PROCESS)


print("\n")
print("=" * 68)
print("🚀 GH ARCHIVE → BRONZE INGESTION")
print("=" * 68)

print("📅 Processing : 2024-02-01 → 2024-02-07")
print(f"📦 Batches    : {total_days} days")
print(f"🥉 Target     : {BRONZE_TABLE}")

print("=" * 68)


# ============================================================
# PROCESS EACH DAY
# ============================================================

for batch_number, day in enumerate(
    DAYS_TO_PROCESS,
    start=1
):

    # --------------------------------------------------------
    # Build date and S3 path
    # --------------------------------------------------------

    date_string = (
        f"{YEAR}-{MONTH}-{day:02d}"
    )

    day_path = (
        f"{RAW_BASE_PATH}"
        f"year={YEAR}/"
        f"month={MONTH}/"
        f"day={day:02d}/"
    )


    print("\n")
    print("=" * 68)

    print(
        f"📦 BATCH {batch_number}/{total_days}"
        f" | {date_string}"
    )

    print("=" * 68)


    # ========================================================
    # DEFINE SPARK TRANSFORMATIONS
    #
    # These operations are lazy.
    # No S3 data is processed yet.
    # ========================================================

    raw_df = (
        spark.read
        .option(
            "recursiveFileLookup",
            "true"
        )
        .text(day_path)
    )


    bronze_df = (
        raw_df
        .select(

            # Original JSON event
            col("value")
            .alias("raw_json"),


            # Full S3 source path
            col("_metadata.file_path")
            .alias("_source_file"),


            # Original archive name
            col("_metadata.file_name")
            .alias("_source_file_name"),


            # Batch date
            lit(date_string)
            .cast("date")
            .alias("_source_date"),


            # Databricks ingestion timestamp
            current_timestamp()
            .alias("_ingested_at")
        )
    )


    # ========================================================
    # SPARK ACTIONS
    #
    # Actions are kept inside try/except so runtime failures
    # can actually be caught.
    # ========================================================

    try:

        # ----------------------------------------------------
        # CHECK IF THIS DATE ALREADY EXISTS
        # ----------------------------------------------------

        existing_files = (
            spark.table(BRONZE_TABLE)
            .filter(
                col("_source_date")
                == lit(date_string).cast("date")
            )
            .select("_source_file_name")
            .distinct()
            .count()
        )


        if existing_files > 0:

            skipped_days += 1

            print(
                f"⏭️  SKIPPED | {date_string}"
            )

            print(
                f"📁 Bronze already contains "
                f"{existing_files} source files"
            )

            print(
                "💡 No data was written."
            )

            continue


        # ----------------------------------------------------
        # INGEST INTO BRONZE
        # ----------------------------------------------------

        print(
            "☁️  Reading 24 hourly archives from S3..."
        )

        print(
            "🥉 Writing events to Bronze Delta table..."
        )


        (
            bronze_df
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(BRONZE_TABLE)
        )


        # ----------------------------------------------------
        # VALIDATE THE WRITTEN BATCH
        # ----------------------------------------------------

        print(
            "🔍 Validating ingested batch..."
        )


        batch_stats = (
            spark.table(BRONZE_TABLE)
            .filter(
                col("_source_date")
                == lit(date_string).cast("date")
            )
            .agg(

                count("*")
                .alias("event_count"),

                countDistinct(
                    "_source_file_name"
                )
                .alias("source_file_count")
            )
            .first()
        )


        event_count = (
            batch_stats["event_count"]
        )

        source_file_count = (
            batch_stats["source_file_count"]
        )


        successful_days += 1


        # ----------------------------------------------------
        # SUCCESS OUTPUT
        # ----------------------------------------------------

        progress = (
            batch_number / total_days
        ) * 100


        print(
            f"✅ INGESTED      | {date_string}"
        )

        print(
            f"📁 Source files  | "
            f"{source_file_count}/24"
        )

        print(
            f"📊 Events loaded | "
            f"{event_count:,}"
        )

        print(
            f"📈 Progress      | "
            f"{batch_number}/{total_days} "
            f"({progress:.0f}%)"
        )


        # ----------------------------------------------------
        # FILE COUNT VALIDATION
        # ----------------------------------------------------

        if source_file_count == 24:

            print(
                "✅ Batch validation passed"
            )

        else:

            print(
                "⚠️  WARNING | Expected 24 source files "
                f"but found {source_file_count}"
            )


    # ========================================================
    # ERROR HANDLING
    # ========================================================

    except Exception as error:

        failed_days += 1

        failed_batches.append(
            date_string
        )


        print(
            f"❌ FAILED | {date_string}"
        )

        print(
            f"🔎 Error  | {error}"
        )

        print(
            "➡️  Continuing with the next batch..."
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 68)
print("🏁 BRONZE INGESTION COMPLETE")
print("=" * 68)

print(
    f"📅 Total batches : {total_days}"
)

print(
    f"✅ Successful    : {successful_days}"
)

print(
    f"⏭️  Skipped       : {skipped_days}"
)

print(
    f"❌ Failed        : {failed_days}"
)


if failed_batches:

    print(
        "⚠️  Failed dates  : "
        + ", ".join(failed_batches)
    )


print(
    f"🥉 Target table  : {BRONZE_TABLE}"
)

print("=" * 68)



🚀 GH ARCHIVE → BRONZE INGESTION
📅 Processing : 2024-02-01 → 2024-02-07
📦 Batches    : 7 days
🥉 Target     : github_lakehouse.bronze.github_events_raw


📦 BATCH 1/7 | 2024-02-01
☁️  Reading 24 hourly archives from S3...
🥉 Writing events to Bronze Delta table...
🔍 Validating ingested batch...
✅ INGESTED      | 2024-02-01
📁 Source files  | 24/24
📊 Events loaded | 5,741,252
📈 Progress      | 1/7 (14%)
✅ Batch validation passed


📦 BATCH 2/7 | 2024-02-02
☁️  Reading 24 hourly archives from S3...
🥉 Writing events to Bronze Delta table...
🔍 Validating ingested batch...
✅ INGESTED      | 2024-02-02
📁 Source files  | 24/24
📊 Events loaded | 5,673,054
📈 Progress      | 2/7 (29%)
✅ Batch validation passed


📦 BATCH 3/7 | 2024-02-03
☁️  Reading 24 hourly archives from S3...
🥉 Writing events to Bronze Delta table...
🔍 Validating ingested batch...
✅ INGESTED      | 2024-02-03
📁 Source files  | 24/24
📊 Events loaded | 4,985,482
📈 Progress      | 3/7 (43%)
✅ Batch validation passed


📦 BATCH 4/7 | 2

# 🔍 Bronze Validation

After ingestion finishes, the complete Bronze dataset is validated.

The checks include:

- total number of GitHub events
- number of events per day
- number of source archive files per day
- total number of unique GH Archive source files

A complete seven-day ingestion should contain:

`7 days × 24 files = 168 source files`

##Load Bronze For Validation

In [0]:
bronze_df = spark.table(
    BRONZE_TABLE
)

print(
    "✅ Bronze Delta table loaded "
    "for validation."
)

✅ Bronze Delta table loaded for validation.


##Total Events

In [0]:
total_events = (
    bronze_df
    .count()
)


print("=" * 60)
print("🥉 BRONZE TABLE SUMMARY")
print("=" * 60)

print(
    f"📊 Total GitHub events | "
    f"{total_events:,}"
)

print("=" * 60)

🥉 BRONZE TABLE SUMMARY
📊 Total GitHub events | 38,555,222


##Events and source files by day

In [0]:
daily_summary = (
    bronze_df
    .groupBy("_source_date")
    .agg(

        count("*")
        .alias("event_count"),

        countDistinct(
            "_source_file_name"
        )
        .alias("source_files")
    )
    .orderBy("_source_date")
)


display(
    daily_summary
)

_source_date,event_count,source_files
2024-02-01,5741252,24
2024-02-02,5673054,24
2024-02-03,4985482,24
2024-02-04,4956032,24
2024-02-05,5589778,24
2024-02-06,5774385,24
2024-02-07,5835239,24


##Verify all 168 source files

In [0]:
total_source_files = (
    bronze_df
    .select("_source_file_name")
    .distinct()
    .count()
)


print("=" * 60)
print("📁 SOURCE FILE VALIDATION")
print("=" * 60)

print(
    f"Expected files : 168"
)

print(
    f"Bronze files   : {total_source_files}"
)


if total_source_files == 168:

    print(
        "✅ All 168 GH Archive files were ingested."
    )

else:

    print(
        "⚠️ Source file count does not match "
        "the expected 168 files."
    )


print("=" * 60)

📁 SOURCE FILE VALIDATION
Expected files : 168
Bronze files   : 168
✅ All 168 GH Archive files were ingested.


##Inspect sample Bronze events

In [0]:
display(
    bronze_df
    .select(
        "raw_json",
        "_source_file_name",
        "_source_date",
        "_ingested_at"
    )
    .limit(20)
)

raw_json _source_file_name _source_date _ingested_at {"id":"35294794163","type":"DeleteEvent","actor":{"id":19267812,"login":"tatsutakein","display_login":"tatsutakein","gravatar_id":"","url":"https://api.github.com/users/tatsutakein","avatar_url":"https://avatars.githubusercontent.com/u/19267812?"},"repo":{"id":674848124,"name":"tatsutakein/zenn-contents","url":"https://api.github.com/repos/tatsutakein/zenn-contents"},"payload":{"ref":"dependabot/npm_and_yarn/zenn-cli-0.1.152","ref_type":"branch","pusher_type":"user"},"public":true,"created_at":"2024-02-01T00:00:00Z"} 2024-02-01-0.json.gz 2024-02-01 2026-09-19T14:18:16.233Z {"id":"35294794178","type":"PullRequestEvent","actor":{"id":41898282,"login":"github-actions[bot]","display_login":"github-actions","gravatar_id":"","url":"https://api.github.com/users/github-actions[bot]","avatar_url":"https://avatars.githubusercontent.com/u/41898282?"},"repo":{"id":711307169,"name":"CodeEditorLand/LandEdgeDebug2","url":"https://api.github.com/repos/CodeEditorLand/LandEdgeDebug2"},"payload":{"action":"closed","number":15,"pull_request":{"url":"https://api.github.com/repos/CodeEditorLand/LandEdgeDebug2/pulls/15","id":1705030234,"node_id":"PR_kwDOKmWvoc5loLJa","html_url":"https://github.com/CodeEditorLand/LandEdgeDebug2/pull/15","diff_url":"https://github.com/CodeEditorLand/LandEdgeDebug2/pull/15.diff","patch_url":"https://github.com/CodeEditorLand/LandEdgeDebug2/pull/15.patch","issue_url":"https://api.github.com/repos/CodeEditorLand/LandEdgeDebug2/issues/15","number":15,"state":"closed","locked":false,"title":"Bump tar from 4.4.13 to 4.4.19","user":{"login":"dependabot[bot]","id":49699333,"node_id":"MDM6Qm90NDk2OTkzMzM=","avatar_url":"https://avatars.githubusercontent.com/in/29110?v=4","gravatar_id":"","url":"https://api.github.com/users/dependabot%5Bbot%5D","html_url":"https://github.com/apps/dependabot","followers_url":"https://api.github.com/users/dependabot%5Bbot%5D/followers","following_url":"https://api.github.com/users/dependabot%5Bbot%5D/following{/other_user}","gists_url":"https://api.github.com/users/dependabot%5Bbot%5D/gists{/gist_id}","starred_url":"https://api.github.com/users/dependabot%5Bbot%5D/starred{/owner}{/repo}","subscriptions_url":"https://api.github.com/users/dependabot%5Bbot%5D/subscriptions","organizations_url":"https://api.github.com/users/dependabot%5Bbot%5D/orgs","repos_url":"https://api.github.com/users/dependabot%5Bbot%5D/repos","events_url":"https://api.github.com/users/dependabot%5Bbot%5D/events{/privacy}","received_events_url":"https://api.github.com/users/dependabot%5Bbot%5D/received_events","type":"Bot","site_admin":false},"body":"Bumps [tar](https://github.com/isaacs/node-tar) from 4.4.13 to 4.4.19.\n \n Commits \n \n 9a6faa0 4.4.19 \n 70ef812 drop dirCache for symlink on all platforms \n 3e35515 4.4.18 \n 52b09e3 fix: prevent path escape using drive-relative paths \n bb93ba2 fix: reserve paths properly for unicode, windows \n 2f1bca0 fix: prune dirCache properly for unicode, windows \n 9bf70a8 4.4.17 \n 6aafff0 fix: skip extract if linkpath is stripped entirely \n 5c5059a fix: reserve paths case-insensitively \n fd6accb 4.4.16 \n Additional commits viewable in compare view \n \n \n \n\n\n[![Dependabot compatibility score](https://dependabot-badges.githubapp.com/badges/compatibility_score?dependency-name=tar&package-manager=npm_and_yarn&previous-version=4.4.13&new-version=4.4.19)](https://docs.github.com/en/github/managing-security-vulnerabilities/about-dependabot-security-updates#about-compatibility-scores)\n\nDependabot will resolve any conflicts with this PR as long as you don't alter it yourself. You can also trigger a rebase manually by commenting `@dependabot rebase`.\n\n[//]: # (dependabot-automerge-start)\n[//]: # (dependabot-automerge-end)\n\n---\n\n \n Dependabot commands and options \n \n\nYou can trigger Dependabot actions by commenting on this PR:\n- `@dependabot rebase` will rebase this PR\n- `@dependabot recreate` will recreate this PR, ov

# ✅ Bronze Layer Complete

The raw GH Archive dataset has now been ingested from Amazon S3 into
a Delta Lake Bronze table.

## Bronze Table

`github_lakehouse.bronze.github_events_raw`

## Final Bronze Schema

```text
root
 |-- raw_json: string
 |-- _source_file: string
 |-- _source_file_name: string
 |-- _source_date: date
 |-- _ingested_at: timestamp